# Notebook 07 — Spectral Graph Partitioning

For specific targets identified as split candidates (high centrality, large, coupling separate domains),
find the mathematically optimal way to divide their source files into new targets.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from scipy.sparse.linalg import eigsh
from scipy.sparse import csgraph
from sklearn.cluster import KMeans

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from build_optimiser.graph import load_graph, attach_metrics, critical_path, node_centrality
from build_optimiser.simulation import rebuild_cost, simulate_split
from build_optimiser.config import load_config

cfg = load_config(PROJECT_ROOT / "config.yaml")

file_metrics = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "file_metrics.parquet")
target_metrics = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "target_metrics.parquet")
edge_list = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "edge_list.parquet")

G = load_graph(str(PROJECT_ROOT / "data" / "raw" / "dot"))
attach_metrics(G, target_metrics)

print(f"Files: {len(file_metrics)}, Targets: {len(target_metrics)}, Edges: {len(edge_list)}")

## Identify Split Candidates

Targets with high betweenness centrality, large file counts, and that couple separate domains.

In [ ]:
# Compute betweenness centrality
centrality = node_centrality(G)

# Merge with target metrics
split_candidates = target_metrics.copy()
split_candidates['betweenness_centrality'] = split_candidates['cmake_target'].map(centrality).fillna(0)

# Score: high centrality + large size + many files
split_candidates['split_score'] = (
    split_candidates['betweenness_centrality'] * 0.4 +
    (split_candidates['file_count'] / split_candidates['file_count'].max()) * 0.3 +
    (split_candidates['compile_time_sum_ms'] / split_candidates['compile_time_sum_ms'].max()) * 0.3
)

# Filter: only targets with multiple files are splittable
split_candidates = split_candidates[split_candidates['file_count'] > 1]
split_candidates = split_candidates.sort_values('split_score', ascending=False)

print("Top split candidates:")
split_candidates[['cmake_target', 'file_count', 'compile_time_sum_ms',
                   'betweenness_centrality', 'split_score']].head(15)

## Build Intra-Target File-Level Dependency Graph

For each split candidate, build a graph where nodes are source files and edges are `#include` relationships.

In [ ]:
def build_file_graph(target_name, file_metrics_df, header_depth_dir=None):
    """Build a file-level dependency graph within a target.
    
    Uses header inclusion data if available, otherwise falls back to
    co-change frequency from git history.
    """
    target_files = file_metrics_df[file_metrics_df['cmake_target'] == target_name]
    file_list = target_files['source_file'].tolist()
    
    fg = nx.Graph()
    for f in file_list:
        fg.add_node(f, **target_files[target_files['source_file'] == f].iloc[0].to_dict())
    
    # Try to infer edges from include relationships
    # Heuristic: files in the same directory are more likely connected
    from pathlib import Path as P
    for i, f1 in enumerate(file_list):
        for f2 in file_list[i+1:]:
            p1, p2 = P(f1), P(f2)
            # Same directory = likely related
            if p1.parent == p2.parent:
                fg.add_edge(f1, f2, weight=2.0)
            # Matching .cpp/.h pairs
            elif p1.stem == p2.stem:
                fg.add_edge(f1, f2, weight=10.0)  # Strong coupling
    
    # Add co-change weights from git history
    if 'git_commit_count' in target_files.columns:
        for i, f1 in enumerate(file_list):
            for f2 in file_list[i+1:]:
                if fg.has_edge(f1, f2):
                    # Weight by co-change frequency
                    c1 = target_files[target_files['source_file'] == f1]['git_commit_count'].iloc[0]
                    c2 = target_files[target_files['source_file'] == f2]['git_commit_count'].iloc[0]
                    cochange = min(c1, c2)  # Approximation
                    fg[f1][f2]['weight'] += cochange
    
    return fg

# Build file graphs for top candidates
top_candidates = split_candidates.head(5)['cmake_target'].tolist()
file_graphs = {}
for target in top_candidates:
    fg = build_file_graph(target, file_metrics)
    file_graphs[target] = fg
    print(f"{target}: {fg.number_of_nodes()} files, {fg.number_of_edges()} edges")

## Spectral Partitioning

Compute the graph Laplacian and its Fiedler vector to find an optimal 2-way partition.

In [ ]:
def spectral_partition(fg, n_parts=2):
    """Partition a file graph using spectral methods.
    
    For 2-way: use the Fiedler vector (sign gives partition).
    For k-way: use first k eigenvectors + k-means.
    """
    if fg.number_of_nodes() < 2:
        return {n: 0 for n in fg.nodes()}
    
    nodes = list(fg.nodes())
    n = len(nodes)
    
    # Build adjacency and Laplacian
    A = nx.adjacency_matrix(fg, nodelist=nodes, weight='weight').toarray().astype(float)
    D = np.diag(A.sum(axis=1))
    L = D - A
    
    if n_parts == 2:
        # Fiedler vector: second-smallest eigenvector of Laplacian
        k = min(2, n - 1)
        eigenvalues, eigenvectors = eigsh(L.astype(float), k=k, which='SM')
        fiedler = eigenvectors[:, -1]  # Second-smallest
        
        partition = {}
        for i, node in enumerate(nodes):
            partition[node] = 0 if fiedler[i] < 0 else 1
    else:
        # k-way: use first k eigenvectors + k-means
        k = min(n_parts, n - 1)
        eigenvalues, eigenvectors = eigsh(L.astype(float), k=k, which='SM')
        
        kmeans = KMeans(n_clusters=n_parts, random_state=42, n_init=10)
        labels = kmeans.fit_predict(eigenvectors)
        
        partition = {}
        for i, node in enumerate(nodes):
            partition[node] = int(labels[i])
    
    return partition

# Apply spectral partitioning to each candidate
spectral_results = {}
for target, fg in file_graphs.items():
    if fg.number_of_nodes() >= 2:
        partition = spectral_partition(fg, n_parts=2)
        spectral_results[target] = partition
        groups = {}
        for node, part in partition.items():
            groups.setdefault(part, []).append(node)
        print(f"\n{target}:")
        for part, files in groups.items():
            print(f"  Partition {part}: {len(files)} files")

## METIS Partitioning (Comparison)

In [ ]:
def metis_partition(fg, n_parts=2):
    """Partition using METIS via pymetis if available."""
    try:
        import pymetis
    except ImportError:
        print("pymetis not available, skipping METIS partitioning")
        return None
    
    nodes = list(fg.nodes())
    if len(nodes) < 2:
        return {n: 0 for n in nodes}
    
    node_idx = {n: i for i, n in enumerate(nodes)}
    
    # Build adjacency list for pymetis
    adjacency = []
    for node in nodes:
        neighbors = [node_idx[n] for n in fg.neighbors(node) if n in node_idx]
        adjacency.append(np.array(neighbors, dtype=np.int32))
    
    # Node weights based on compile time
    node_weights = []
    for node in nodes:
        ct = fg.nodes[node].get('compile_time_ms', 1)
        node_weights.append(max(1, int(ct)))
    
    n_cuts, membership = pymetis.part_graph(
        n_parts, adjacency=adjacency, vweights=node_weights
    )
    
    return {nodes[i]: membership[i] for i in range(len(nodes))}

# Compare METIS with spectral
metis_results = {}
for target, fg in file_graphs.items():
    if fg.number_of_nodes() >= 2:
        partition = metis_partition(fg, n_parts=2)
        if partition:
            metis_results[target] = partition

## Constraint Handling

Contract .cpp/.h pairs into single nodes before partitioning.

In [ ]:
def contract_header_pairs(fg):
    """Contract .cpp and matching .h files into single nodes."""
    from pathlib import Path as P
    
    contracted = fg.copy()
    nodes = list(contracted.nodes())
    
    # Find .cpp/.h pairs by stem
    stems = {}
    for n in nodes:
        stem = P(n).stem
        stems.setdefault(stem, []).append(n)
    
    contractions = {}  # contracted_name -> [original files]
    for stem, files in stems.items():
        cpp_files = [f for f in files if P(f).suffix in ('.cpp', '.cc', '.cxx')]
        h_files = [f for f in files if P(f).suffix in ('.h', '.hpp', '.hxx')]
        if cpp_files and h_files:
            merged_name = cpp_files[0]  # Use cpp as representative
            contractions[merged_name] = cpp_files + h_files
            # Contract: merge h into cpp node
            for h in h_files:
                if h in contracted and merged_name in contracted:
                    contracted = nx.contracted_nodes(contracted, merged_name, h, self_loops=False)
    
    return contracted, contractions

# Apply to top candidate
if file_graphs:
    target = top_candidates[0]
    fg = file_graphs[target]
    contracted_fg, contractions = contract_header_pairs(fg)
    print(f"{target}: {fg.number_of_nodes()} files -> {contracted_fg.number_of_nodes()} contracted nodes")
    if contractions:
        print(f"Contracted pairs: {len(contractions)}")

## Partition Evaluation

In [ ]:
def evaluate_partition(fg, partition, file_metrics_df, target_name):
    """Evaluate a partition: cross-partition edges, compile time balance."""
    groups = {}
    for node, part in partition.items():
        groups.setdefault(part, []).append(node)
    
    # Cross-partition edges (these become inter-target dependencies)
    cross_edges = sum(
        1 for u, v in fg.edges()
        if partition.get(u, -1) != partition.get(v, -1)
    )
    
    # Compile time balance
    target_files = file_metrics_df[file_metrics_df['cmake_target'] == target_name]
    part_times = {}
    for part, files in groups.items():
        mask = target_files['source_file'].isin(files)
        ct = target_files.loc[mask, 'compile_time_ms'].sum()
        part_times[part] = ct
    
    times = list(part_times.values())
    balance = min(times) / max(times) if max(times) > 0 else 1.0
    
    return {
        'cross_partition_edges': cross_edges,
        'total_edges': fg.number_of_edges(),
        'partition_sizes': {k: len(v) for k, v in groups.items()},
        'partition_compile_times': part_times,
        'balance_ratio': balance,
    }

# Evaluate all spectral partitions
print("Spectral Partition Evaluation:")
print("=" * 60)
for target in spectral_results:
    if target in file_graphs:
        eval_result = evaluate_partition(
            file_graphs[target], spectral_results[target], file_metrics, target
        )
        print(f"\n{target}:")
        print(f"  Cross-partition edges: {eval_result['cross_partition_edges']}/{eval_result['total_edges']}")
        print(f"  Partition sizes: {eval_result['partition_sizes']}")
        print(f"  Compile time balance: {eval_result['balance_ratio']:.2f}")

## Visualise Partitions

In [ ]:
# Visualise the partition for the top candidate
if spectral_results:
    target = top_candidates[0]
    if target in file_graphs and target in spectral_results:
        fg = file_graphs[target]
        partition = spectral_results[target]
        
        fig, ax = plt.subplots(1, 1, figsize=(12, 8))
        
        pos = nx.spring_layout(fg, seed=42)
        colors = [partition.get(n, 0) for n in fg.nodes()]
        
        # Draw cross-partition edges in red, same-partition in gray
        cross_edges = [(u, v) for u, v in fg.edges() if partition.get(u) != partition.get(v)]
        same_edges = [(u, v) for u, v in fg.edges() if partition.get(u) == partition.get(v)]
        
        nx.draw_networkx_edges(fg, pos, edgelist=same_edges, edge_color='lightgray', ax=ax)
        nx.draw_networkx_edges(fg, pos, edgelist=cross_edges, edge_color='red', style='dashed', ax=ax)
        
        nx.draw_networkx_nodes(fg, pos, node_color=colors, cmap=plt.cm.Set1, ax=ax, node_size=200)
        
        # Short labels
        from pathlib import Path as P
        labels = {n: P(n).name for n in fg.nodes()}
        nx.draw_networkx_labels(fg, pos, labels, font_size=6, ax=ax)
        
        ax.set_title(f'Spectral Partition: {target}')
        plt.tight_layout()
        plt.savefig(str(PROJECT_ROOT / 'data' / 'results' / f'partition_{target}.png'), dpi=150, bbox_inches='tight')
        plt.show()